# FINBRT classification and embedding

In [6]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

MODEL_NAME = "ProsusAI/finbert"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Shared tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Base FinBERT for embeddings
base_model = AutoModel.from_pretrained(MODEL_NAME)
base_model.to(device)
base_model.eval()

# FinBERT classifier for sentiment
clf_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
clf_model.to(device)
clf_model.eval()


Using device: cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [2]:
import pandas as pd

def add_text_column(df: pd.DataFrame) -> pd.DataFrame:
    # prefer summary; fallback to title; drop rows with nothing
    text = df["Lsa_summary"].fillna("") + " " + df["Article_title"].fillna("")
    text = text.str.strip()
    df = df.assign(text=text)
    df = df[df["text"] != ""]
    return df


In [7]:
class NewsTextDataset(Dataset):
    def __init__(self, texts):
        self.texts = list(texts)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]

def collate_batch(batch_texts, max_len=128):
    return tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )


In [8]:
def run_finbert_dual_on_split(
    input_path: str,
    output_path: str,
    batch_size: int = 32,
    max_len: int = 128
):
    print(f"Loading {input_path} ...")
    df = pd.read_parquet(input_path)
    print("Original shape:", df.shape)

    # Ensure Date is datetime
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", utc=True)
    df = df.dropna(subset=["Date"])

    # Build text column (summary + title)
    df = add_text_column(df)
    df = df.reset_index(drop=True)
    print("After dropping empty text:", df.shape)

    dataset = NewsTextDataset(df["text"])
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    all_logits = []
    all_embeddings = []

    torch.cuda.empty_cache()

    for i, batch_texts in enumerate(loader):
        if i % 200 == 0:
            print(f"Batch {i}/{len(loader)}")

        encodings = collate_batch(batch_texts, max_len=max_len)
        encodings = {k: v.to(device) for k, v in encodings.items()}

        with torch.no_grad():
            # Base model → embeddings
            base_out = base_model(**encodings, return_dict=True)
            last_hidden = base_out.last_hidden_state          # (B, seq_len, 768)
            cls_emb = last_hidden[:, 0, :]                    # CLS token

            # Classifier → sentiment logits
            clf_out = clf_model(**encodings, return_dict=True)
            logits = clf_out.logits                           # (B, 3)

        all_embeddings.append(cls_emb.cpu().numpy().astype("float32"))
        all_logits.append(logits.cpu().numpy().astype("float32"))

    # Stack all batches
    all_embeddings = np.concatenate(all_embeddings, axis=0)   # (N, 768)
    all_logits = np.concatenate(all_logits, axis=0)           # (N, 3)

    print("Embeddings shape:", all_embeddings.shape)
    print("Logits shape:", all_logits.shape)

    # Convert logits → probs → sentiment
    probs = torch.softmax(torch.tensor(all_logits), dim=1).numpy()
    neg_prob = probs[:, 0]
    neu_prob = probs[:, 1]
    pos_prob = probs[:, 2]
    sentiment_score = pos_prob - neg_prob       # scalar sentiment
    sentiment_label = probs.argmax(axis=1)      # 0=neg,1=neu,2=pos

    # Attach sentiment columns
    df["finbert_neg_prob"] = neg_prob
    df["finbert_neu_prob"] = neu_prob
    df["finbert_pos_prob"] = pos_prob
    df["finbert_sentiment_score"] = sentiment_score
    df["finbert_sentiment_label"] = sentiment_label

    # Attach embedding dims as separate columns
    emb_dim = all_embeddings.shape[1]
    for j in range(emb_dim):
        df[f"finbert_emb_{j}"] = all_embeddings[:, j]

    print("Final DF shape with features:", df.shape)
    df.to_parquet(output_path)
    print(f"Saved: {output_path}")


In [9]:
run_finbert_dual_on_split(
    "../data/model/train_news.parquet",
    "../data/model/train_news_finbert.parquet",
    batch_size=32,
    max_len=128
)

run_finbert_dual_on_split(
    "../data/model/val_news.parquet",
    "../data/model/val_news_finbert.parquet",
    batch_size=32,
    max_len=128
)

run_finbert_dual_on_split(
    "../data/model/test_news.parquet",
    "../data/model/test_news_finbert.parquet",
    batch_size=32,
    max_len=128
)


Loading ../data/model/train_news.parquet ...
Original shape: (921967, 5)
After dropping empty text: (921967, 6)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Batch 0/28812
Batch 200/28812
Batch 400/28812
Batch 600/28812
Batch 800/28812
Batch 1000/28812
Batch 1200/28812
Batch 1400/28812
Batch 1600/28812
Batch 1800/28812
Batch 2000/28812
Batch 2200/28812
Batch 2400/28812
Batch 2600/28812
Batch 2800/28812
Batch 3000/28812
Batch 3200/28812
Batch 3400/28812
Batch 3600/28812
Batch 3800/28812
Batch 4000/28812
Batch 4200/28812
Batch 4400/28812
Batch 4600/28812
Batch 4800/28812
Batch 5000/28812
Batch 5200/28812
Batch 5400/28812
Batch 5600/28812
Batch 5800/28812
Batch 6000/28812
Batch 6200/28812
Batch 6400/28812
Batch 6600/28812
Batch 6800/28812
Batch 7000/28812
Batch 7200/28812
Batch 7400/28812
Batch 7600/28812
Batch 7800/28812
Batch 8000/28812
Batch 8200/28812
Batch 8400/28812
Batch 8600/28812
Batch 8800/28812
Batch 9000/28812
Batch 9200/28812
Batch 9400/28812
Batch 9600/28812
Batch 9800/28812
Batch 10000/28812
Batch 10200/28812
Batch 10400/28812
Batch 10600/28812
Batch 10800/28812
Batch 11000/28812
Batch 11200/28812
Batch 11400/28812
Batch 11600/2

/tmp/ipykernel_2709266/581130911.py:79: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"finbert_emb_{j}"] = all_embeddings[:, j]
/tmp/ipykernel_2709266/581130911.py:79: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"finbert_emb_{j}"] = all_embeddings[:, j]
/tmp/ipykernel_2709266/581130911.py:79: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-f

Final DF shape with features: (921967, 779)
Saved: ../data/model/train_news_finbert.parquet
Loading ../data/model/val_news.parquet ...
Original shape: (112846, 5)
After dropping empty text: (112846, 6)


...

Final DF shape with features: (112846, 779)
Saved: ../data/model/val_news_finbert.parquet
Loading ../data/model/test_news.parquet ...
Original shape: (159629, 5)
After dropping empty text: (159629, 6)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Batch 0/4989
Batch 200/4989
Batch 400/4989
Batch 600/4989
Batch 800/4989
Batch 1000/4989
Batch 1200/4989
Batch 1400/4989
Batch 1600/4989
Batch 1800/4989
Batch 2000/4989
Batch 2200/4989
Batch 2400/4989
Batch 2600/4989
Batch 2800/4989
Batch 3000/4989
Batch 3200/4989
Batch 3400/4989
Batch 3600/4989
Batch 3800/4989
Batch 4000/4989
Batch 4200/4989
Batch 4400/4989
Batch 4600/4989
Batch 4800/4989
Embeddings shape: (159629, 768)
Logits shape: (159629, 3)


/tmp/ipykernel_2709266/581130911.py:79: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"finbert_emb_{j}"] = all_embeddings[:, j]
/tmp/ipykernel_2709266/581130911.py:79: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"finbert_emb_{j}"] = all_embeddings[:, j]
/tmp/ipykernel_2709266/581130911.py:79: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-f

Final DF shape with features: (159629, 779)
Saved: ../data/model/test_news_finbert.parquet


In [24]:
import pandas as pd

def aggregate_daily_features(input_path, output_path):
    print(f"\nLoading {input_path} ...")
    df = pd.read_parquet(input_path)
    
    # Ensure date format
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", utc=True)
    df = df.dropna(subset=["Date"])
    print("Shape after cleaning:", df.shape)

    # Identify FinBERT embedding columns
    emb_cols = [c for c in df.columns if c.startswith("finbert_emb_")]
    print(f"Embedding dims detected: {len(emb_cols)}")

    # Sentiment aggregation
    sent_agg = (
        df.groupby(["Date", "Stock_symbol"])
          .agg(
              mean_sentiment=("finbert_sentiment_score", "mean"),
              max_sentiment=("finbert_sentiment_score", "max"),
              min_sentiment=("finbert_sentiment_score", "min"),
              sum_sentiment=("finbert_sentiment_score", "sum"),
              news_count=("finbert_sentiment_score", "count")
          )
    )

    # Embedding aggregation (mean pooling)
    emb_agg = (
        df.groupby(["Date", "Stock_symbol"])[emb_cols]
          .mean()
    )

    # Combine sentiment + embedding features
    daily_features = pd.concat([sent_agg, emb_agg], axis=1).reset_index()
    print("Final daily feature shape:", daily_features.shape)

    # Save
    daily_features.to_parquet(output_path)
    print(f"Saved: {output_path}")


In [26]:
aggregate_daily_features(
    "../data/model/train_news_finbert.parquet",
    "../data/model/train_news_daily_features.parquet"
)



Loading ../data/model/train_news_finbert.parquet ...
Shape after cleaning: (921967, 779)
Embedding dims detected: 768
Final daily feature shape: (398780, 775)
Saved: ../data/model/train_news_daily_features.parquet


In [27]:
aggregate_daily_features(
    "../data/model/val_news_finbert.parquet",
    "../data/model/val_news_daily_features.parquet"
)



Loading ../data/model/val_news_finbert.parquet ...
Shape after cleaning: (112846, 779)
Embedding dims detected: 768
Final daily feature shape: (46756, 775)
Saved: ../data/model/val_news_daily_features.parquet


In [28]:
aggregate_daily_features(
    "../data/model/test_news_finbert.parquet",
    "../data/model/test_news_daily_features.parquet"
)



Loading ../data/model/test_news_finbert.parquet ...
Shape after cleaning: (159629, 779)
Embedding dims detected: 768
Final daily feature shape: (55339, 775)
Saved: ../data/model/test_news_daily_features.parquet


# PAC dimension reduction

In [29]:
import pandas as pd

train = pd.read_parquet("../data/model/train_news_daily_features.parquet")

# Identify embedding columns
emb_cols = [c for c in train.columns if c.startswith("finbert_emb_")]
len(emb_cols)


768

In [30]:
from sklearn.decomposition import PCA

pca_dim = 64
pca = PCA(n_components=pca_dim, random_state=42)

# Fit on train embeddings
pca.fit(train[emb_cols])


,n_components,64
,copy,True
,whiten,False
,svd_solver,'auto'
,tol,0.0
,iterated_power,'auto'
,n_oversamples,10
,power_iteration_normalizer,'auto'
,random_state,42


In [31]:
train_pca = pca.transform(train[emb_cols])
pca_cols = [f"pca_emb_{i}" for i in range(pca_dim)]

train_pca_df = pd.DataFrame(train_pca, columns=pca_cols, index=train.index)


In [32]:
train_reduced = pd.concat([
    train.drop(columns=emb_cols),
    train_pca_df
], axis=1)

train_reduced.to_parquet("../data/model/train_news_daily_pca.parquet")
print("Saved TRAIN PCA file")


Saved TRAIN PCA file


In [33]:
train_reduced

,Date,Stock_symbol,mean_sentiment,max_sentiment,min_sentiment,sum_sentiment,news_count,pca_emb_0,pca_emb_1,pca_emb_2,...,pca_emb_54,pca_emb_55,pca_emb_56,pca_emb_57,pca_emb_58,pca_emb_59,pca_emb_60,pca_emb_61,pca_emb_62,pca_emb_63
0,2014-01-01 00:00:00+00:00,CTSH,0.238317,0.238317,0.238317,0.238317,1,-3.559703,-1.447504,-0.839296,...,0.179844,-1.047858,-0.829812,1.048810,0.464801,-0.622704,-0.530804,-0.195862,0.651855,0.121191
1,2014-01-01 00:00:00+00:00,GPC,0.638470,0.638470,0.638470,0.638470,1,0.756325,1.202744,-1.777061,...,0.303519,-1.179288,-0.656293,0.461733,0.049484,-0.486424,0.442076,-0.012598,0.721352,0.427091
2,2014-01-01 00:00:00+00:00,GRMN,0.865825,0.865825,0.865825,0.865825,1,5.078024,4.010743,-2.381730,...,0.406681,-0.066577,-0.858613,0.525824,-0.207917,-1.138054,-0.135715,0.516298,0.319620,0.557026
3,2014-01-01 00:00:00+00:00,HLT,0.238313,0.238313,0.238313,0.238313,1,-3.559710,-1.447512,-0.839299,...,0.179843,-1.047860,-0.829814,1.048809,0.464799,-0.622702,-0.530802,-0.195861,0.651858,0.121191
4,2014-01-01 00:00:00+00:00,MCHP,0.865825,0.865825,0.865825,0.865825,1,5.078025,4.010744,-2.381730,...,0.406682,-0.066576,-0.858616,0.525824,-0.207916,-1.138054,-0.135715,0.516300,0.319620,0.557027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
398775,2021-12-31 00:00:00+00:00,WMT,-0.059380,0.570769,-0.792475,-0.178141,3,-5.849300,-2.199842,1.876482,...,-0.633156,-0.055679,0.247241,0.334318,-0.019939,0.261622,-0.194300,0.235104,-0.722728,0.034238
398776,2021-12-31 00:00:00+00:00,WST,0.010281,0.010281,0.010281,0.010281,1,-9.003062,-0.736913,4.467481,...,-0.730233,-0.925525,-1.353305,0.602277,0.641416,0.914600,0.819552,-0.545032,0.017053,0.004584
398777,2021-12-31 00:00:00+00:00,WY,-0.007851,-0.007851,-0.007851,-0.007851,1,-5.932696,-1.895413,4.421789,...,0.085252,-0.627794,-1.332258,-0.353745,-0.633953,1.091674,0.780312,-0.497025,0.799557,0.569756
398778,2021-12-31 00:00:00+00:00,XEL,-0.408841,0.033462,-0.851144,-0.817682,2,-12.454595,-1.281286,0.680101,...,-0.485131,0.206023,0.403742,0.006716,0.045933,0.418239,-0.446136,-0.404604,0.063422,0.577922


In [35]:
val = pd.read_parquet("../data/model/val_news_daily_features.parquet")
val_pca = pca.transform(val[emb_cols])
val_pca_df = pd.DataFrame(val_pca, columns=pca_cols, index=val.index)

val_reduced = pd.concat([
    val.drop(columns=emb_cols),
    val_pca_df
], axis=1)

val_reduced.to_parquet("../data/model/val_news_daily_pca.parquet")
print("Saved VAL PCA file")


Saved VAL PCA file


In [36]:
test = pd.read_parquet("../data/model/test_news_daily_features.parquet")
test_pca = pca.transform(test[emb_cols])
test_pca_df = pd.DataFrame(test_pca, columns=pca_cols, index=test.index)

test_reduced = pd.concat([
    test.drop(columns=emb_cols),
    test_pca_df
], axis=1)

test_reduced.to_parquet("../data/model/test_news_daily_pca.parquet")
print("Saved TEST PCA file")


Saved TEST PCA file
